# Advanced RAG - Chunking Strategies, RAG Patterns, Hybrid Search, and Reranking

Prerequisites: As we have completed the basic RAG notebook (setup, basic chunking, FAISS vector store, RAG pipeline, overlapping chunks, citations).

This notebook covers:

1. Advanced Document Chunking Strategies
   - Semantic Chunking
   - Hierarchical Chunking
   - Sentence-Window Chunking
   - Recursive Character Chunking (with tuning)

2. Advanced RAG Patterns
   - Simple RAG (recap)
   - RAG with Conversation Memory
   - HyDE (Hypothetical Document Embeddings)
   - Branched RAG (Query Routing)
   - Multi-Query RAG
   - Multimodal RAG (concept + implementation with Gemini)

3. Hybrid Search and Retrieval
   - Dense retrieval (FAISS)
   - Sparse retrieval (BM25)
   - Combining both with Reciprocal Rank Fusion

4. Reranking Models
   - What reranking is and why it matters
   - Cross-encoder reranking with sentence-transformers
   - Full pipeline: Retrieve -> Rerank -> Generate

---

Models used:
- LLM: `llama-3.1-8b-instant` via Groq 
- Embeddings: `all-MiniLM-L6-v2` via sentence-transformers 
- Multimodal LLM: `gemini-1.5-flash` via Google Generative AI (can use free tier)
- Reranker: `cross-encoder/ms-marco-MiniLM-L-6-v2` via sentence-transformers 

## 1. Environment Setup

In [ ]:
# Install all dependencies needed for this notebook.
# rank_bm25       -> sparse retrieval (BM25 keyword search)
# google-generativeai -> Gemini API for multimodal RAG
# Pillow          -> image handling for multimodal RAG

# !pip install langchain langchain-groq langchain-community langchain-huggingface sentence-transformers faiss-cpu rank_bm25 google-generativeai Pillow python-dotenv langchain_experimental
# !pip install --upgrade langchain langchain-community langchain-text-splitters



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load environment variables from .env file.

from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")  # Optional - only for multimodal section

print("Environment loaded.")
print(f"GROQ_API_KEY present: {bool(GROQ_API_KEY)}")
print(f"GOOGLE_API_KEY present: {bool(GOOGLE_API_KEY)}")

Environment loaded.
GROQ_API_KEY present: True
GOOGLE_API_KEY present: True


In [3]:
# Initialize the LLM and embedding model that we will reuse throughout this notebook.
# These are the same models from our previous notebook so that we keep consistency.

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# Groq LLM - llama-3.1-8b-instant is fast and free
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,  # Low temperature for factual RAG answers
    max_tokens=1000,
)

# HuggingFace embedding model - runs locally
# all-MiniLM-L6-v2: 384-dimensional embeddings, fast and accurate for semantic search
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("LLM and embedding model initialized.")

d:\Work\Learning\Topics_by_Abhay_Sir\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2112.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM and embedding model initialized.


In [4]:
# We need sample documents to demonstrate all chunking and retrieval strategies.
# We will use a multi-paragraph text about AI and machine learning as our corpus.
# In production, we will load PDFs, web pages, databases etc.

sample_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural 
intelligence displayed by animals including humans. AI research has been defined as the field of 
study of intelligent agents, which refers to any system that perceives its environment and takes 
actions that maximize its chance of achieving its goals.

Machine learning (ML) is a subset of artificial intelligence. ML algorithms build a model based 
on sample data, known as training data, in order to make predictions or decisions without being 
explicitly programmed to do so. ML is used in a wide variety of applications, such as in medicine, 
email filtering, speech recognition, agriculture, and computer vision.

Deep learning is part of a broader family of machine learning methods based on artificial neural 
networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. 
Deep learning architectures such as deep neural networks, recurrent neural networks, and convolutional 
neural networks have been applied to fields including computer vision, natural language processing, 
and speech recognition.

Natural Language Processing (NLP) is a subfield of linguistics, computer science, and artificial 
intelligence concerned with the interactions between computers and human language, in particular 
how to program computers to process and analyze large amounts of natural language data. NLP is used 
in sentiment analysis, machine translation, named entity recognition, and question answering.

Retrieval-Augmented Generation (RAG) is an AI framework for retrieving facts from an external 
knowledge base to ground large language models on the most accurate, up-to-date information. 
RAG reduces hallucination by grounding the LLM response in retrieved context rather than relying 
solely on parametric memory. RAG systems combine dense retrieval with generative models.

Vector databases store data as high-dimensional vectors, enabling fast similarity search. 
Popular vector databases include FAISS, Pinecone, Weaviate, Chroma, and Qdrant. They are 
fundamental to RAG systems because they allow semantic search over large document collections. 
FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search.

Large Language Models (LLMs) are neural networks trained on vast amounts of text data. 
They can generate coherent, contextually relevant text. Examples include GPT-4, Claude, Llama, 
Gemini, and Mistral. LLMs suffer from hallucination, outdated knowledge, and context window limits, 
which is why RAG is used to augment them with external, up-to-date knowledge.

Prompt engineering is the practice of designing inputs to AI models to achieve desired outputs. 
Effective prompts are clear, specific, and structured. Techniques include zero-shot prompting, 
few-shot prompting, chain-of-thought prompting, and role prompting. Good prompt engineering 
significantly improves the quality and reliability of LLM outputs.

Embeddings are numerical representations of data in a high-dimensional vector space. 
Similar items have embeddings that are close together in this space, enabling semantic similarity 
comparison. Text embeddings capture the meaning of words and sentences. Models like BERT, 
Sentence-BERT, and OpenAI Ada produce state-of-the-art text embeddings.

Autonomous agents are AI systems that can perceive their environment, make decisions, and take 
actions to achieve goals without continuous human intervention. Modern AI agents use LLMs as their 
reasoning core and are equipped with tools like web search, code execution, and file management. 
Frameworks like LangGraph, AutoGen, and CrewAI are used to build multi-agent systems.
"""

print(f"Sample corpus loaded. Total characters: {len(sample_text)}")

Sample corpus loaded. Total characters: 3739


---

## 2. Advanced Document Chunking Strategies

Chunking is the process of splitting a large document into smaller pieces before storing them in a vector database. The quality of chunking directly affects retrieval quality. Poor chunking means key context gets cut off, leading to bad answers.

We will cover four strategies:

| Strategy | Best For |
|---|---|
| Recursive Character | General purpose, most commonly used |
| Semantic | Topic-based splitting, reduces context breaks |
| Hierarchical | Documents with headers/structure (PDFs, Markdown) |
| Sentence-Window | High precision retrieval with surrounding context |

### 2.1 Recursive Character Text Splitting (Baseline)

This is the most common chunking method. It tries to split on natural boundaries (paragraphs, then sentences, then words) while respecting a max chunk size. We have already used this in the notebook1_rag_basics notebook - here we tune it more carefully.

In [5]:
# RecursiveCharacterTextSplitter tries separators in order: paragraph -> sentence -> word -> character.
# This preserves natural text boundaries as much as possible.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# chunk_size: max characters per chunk
# chunk_overlap: how many characters overlap between consecutive chunks
# This overlap is critical — it ensures context isn't lost at chunk boundaries

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],  # Priority order of split points
    length_function=len,    # Use character count for chunking (default is also len, but can be customized for tokens etc.
)

# Wrap raw text as a LangChain Document with metadata
raw_doc = Document(page_content=sample_text, metadata={"source": "ai_overview"})

recursive_chunks = recursive_splitter.split_documents([raw_doc])

# Add chunk index to metadata for traceability
for i, chunk in enumerate(recursive_chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["chunk_type"] = "recursive"

print(f"Number of chunks: {len(recursive_chunks)}")
print(f"\nFirst chunk (chars: {len(recursive_chunks[0].page_content)}):")
print(recursive_chunks[0].page_content)
print("\nMetadata:", recursive_chunks[0].metadata)

Number of chunks: 10

First chunk (chars: 350):
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural 
intelligence displayed by animals including humans. AI research has been defined as the field of 
study of intelligent agents, which refers to any system that perceives its environment and takes 
actions that maximize its chance of achieving its goals.

Metadata: {'source': 'ai_overview', 'chunk_id': 0, 'chunk_type': 'recursive'}


### 2.2 Semantic Chunking

Instead of splitting on character count, semantic chunking splits on 'meaning shifts'. It computes embeddings for each sentence, then detects where the semantic similarity drops significantly so that drop indicates a topic change, which becomes a split point.

**Why this is better:** Keeps topically related sentences together, even if they are worded differently. Prevents a single idea from being split across two chunks.

In [6]:
# Semantic chunking from LangChain uses embeddings to detect topic boundaries.
# breakpoint_threshold_type="percentile" means: find the top X% largest similarity drops
# and use those as split points. Lower percentile = fewer, larger chunks.

from langchain_experimental.text_splitter import SemanticChunker

# SemanticChunker needs an embedding model to compare sentence similarities
semantic_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",  # Split at top 95th percentile similarity drops
    breakpoint_threshold_amount=95,
)

semantic_chunks = semantic_splitter.split_documents([raw_doc])

for i, chunk in enumerate(semantic_chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["chunk_type"] = "semantic"

print(f"Number of semantic chunks: {len(semantic_chunks)}")
print("\nAll semantic chunks and their sizes:")
for i, chunk in enumerate(semantic_chunks):
    print(f"  Chunk {i}: {len(chunk.page_content)} chars - {chunk.page_content[:80].strip()}...")

Number of semantic chunks: 3

All semantic chunks and their sizes:
  Chunk 0: 2424 chars - Artificial intelligence (AI) is intelligence demonstrated by machines, as oppos...
  Chunk 1: 60 chars - Examples include GPT-4, Claude, Llama, 
Gemini, and Mistral....
  Chunk 2: 1236 chars - LLMs suffer from hallucination, outdated knowledge, and context window limits,...


In [7]:
# Compare: recursive chunks vs semantic chunks
# Recursive chunking may split in the middle of a topic.
# Semantic chunking should keep each paragraph/topic together as a single chunk.

print("--- Recursive Chunking ---")
print(f"Total chunks: {len(recursive_chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in recursive_chunks) // len(recursive_chunks)} chars")

print("\n--- Semantic Chunking ---")
print(f"Total chunks: {len(semantic_chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in semantic_chunks) // len(semantic_chunks)} chars")

# Key insight: 
# Semantic chunks are fewer and larger, but each chunk is topically coherent.
# This often leads to better retrieval because the retrieved chunk contains the full context of a topic.

--- Recursive Chunking ---
Total chunks: 10
Avg chunk size: 371 chars

--- Semantic Chunking ---
Total chunks: 3
Avg chunk size: 1240 chars


### 2.3 Hierarchical Chunking

Hierarchical chunking creates chunks at multiple levels of granularity - e.g., the full paragraph (parent) and individual sentences (children). During retrieval, we match on small children chunks (more precise), but return the full parent chunk as context (more informative).

Contextual Retrieval: During a search, the system retrieves relevant child chunks but can feed the entire parent chunk to the LLM to ensure full context is preserved.

```
Why do we need it? (The Problem)
Small Chunks: Great for finding specific facts, but the LLM gets confused because it lacks the surrounding context (e.g., it finds the answer "Yes," but doesn't know what the question was).
Large Chunks: Great for context, but hard to find. A 2,000-word chunk has too many different topics, which "dilutes" the search and makes it harder for the system to match - your specific question.

How does it work? (The Steps)
Split Big (Parents): Break your document into large, meaningful sections (e.g., 1000 characters).
Split Small (Children): Take those big sections and break them into tiny pieces (e.g., 200 characters).
The Link: Every "child" chunk carries a hidden ID that points back to its "parent."
The Retrieval: When you ask a question, the system looks at the children to find a match. Once it finds the best tiny chunk, it says, "Wait, don't give the LLM just this sentence. Give it the whole Parent section this sentence came from."

A Simple Analogy
Imagine you are looking for a specific quote in a massive library.
The Child Chunk is like the Index at the back of a book. It tells you exactly which page to look at.
The Parent Chunk is the Chapter containing that page.
You search the index (Child) to find the location, but you read the whole chapter (Parent) to understand the full story.
```

**Why this is better:** Small chunks are better for retrieval (precise matching), but larger chunks provide better context for generation. This strategy gets the best of both worlds.

This is also called the **Parent Document Retriever** pattern in LangChain.

In [8]:
# ParentDocumentRetriever from LangChain implements hierarchical chunking.
# It stores:
#   - Parent chunks (large) in an in-memory docstore
#   - Child chunks (small) in a vector store
# When a child chunk is retrieved, the parent chunk is returned as context.

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parent splitter: creates large chunks (full paragraphs/sections)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Large parent chunks
    chunk_overlap=0,
)

# Child splitter: creates small chunks for precise retrieval
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,   # Small child chunks - better for embedding similarity
    chunk_overlap=20,
)

# InMemoryStore holds the parent documents (not embedded, just stored)
parent_docstore = InMemoryStore()

# FAISS vector store holds the child chunk embeddings
# We initialize with a dummy embedding first, then add docs
child_vectorstore = FAISS.from_texts(
    ["placeholder"],  # Temporary placeholder to initialize FAISS
    embedding=embedding_model
)

# Create the hierarchical retriever
hierarchical_retriever = ParentDocumentRetriever(
    vectorstore=child_vectorstore,
    docstore=parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Add documents - this automatically splits into parent/child chunks
hierarchical_retriever.add_documents([raw_doc])

print("Hierarchical retriever built.")

# Test: retrieve with a query
# Notice: even though child chunks are small, the returned docs are full parent chunks
test_query = "What is RAG and how does it work?"
results = hierarchical_retriever.invoke(test_query)

print(f"\nQuery: '{test_query}'")
print(f"Number of results returned: {len(results)}")
print(f"\nRetrieved parent chunk (first result, {len(results[0].page_content)} chars):")
print(results[0].page_content)

Hierarchical retriever built.

Query: 'What is RAG and how does it work?'
Number of results returned: 2

Retrieved parent chunk (first result, 750 chars):
Retrieval-Augmented Generation (RAG) is an AI framework for retrieving facts from an external 
knowledge base to ground large language models on the most accurate, up-to-date information. 
RAG reduces hallucination by grounding the LLM response in retrieved context rather than relying 
solely on parametric memory. RAG systems combine dense retrieval with generative models.

Vector databases store data as high-dimensional vectors, enabling fast similarity search. 
Popular vector databases include FAISS, Pinecone, Weaviate, Chroma, and Qdrant. They are 
fundamental to RAG systems because they allow semantic search over large document collections. 
FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search.


### 2.4 Sentence-Window Chunking

Each sentence is embedded individually (for precise retrieval), but when returned, the sentence is accompanied by a window of surrounding sentences (for context). This is different from hierarchical chunking - here every chunk is a single sentence, and the window is added at retrieval time.

**Why this is useful:** Extremely precise retrieval (single sentence match), with enough surrounding context for the LLM to understand it.

```
How it Works (The "Center-Out" Logic)
    Storage: The document is split into individual sentences. Each sentence is stored in the vector database with its position (index) and metadata.
    Retrieval: When you ask a question, the system finds the single best matching sentence.
    Expansion: Instead of just sending that one sentence to the LLM, the system "looks left and right" to grab number of sentences before and after it.
    Final Prompt: The LLM receives a cohesive paragraph where the "answer" is right in the middle.

Why it's different from Hierarchical Chunking
Hierarchical: You have a pre-defined "Parent" (like a fixed 1000-character box). If your sentence is at the very end of that box, you get the context before it, but none after it.
Sentence-Window: The context is always centered. You get a perfect "buffer" on both sides, regardless of where the sentence sits in the document.

A Simple Analogy
Imagine a Spotlight:
The Search is a laser pointer—it hits one specific spot (the sentence).
The Window is the spotlight—once the laser hits the spot, the spotlight turns on to show you the immediate area around it so you can see what’s happening.

Pro-Tip: When to use it?
Use this when your data is dense with facts (like legal contracts or research papers) where a single sentence holds the answer, but that answer is meaningless without the sentences right next to it.
```

In [9]:
# We implement sentence-window chunking manually.
# Step 1: Split document into individual sentences.
# Step 2: Store each sentence along with its surrounding window in metadata.
# Step 3: Embed each sentence individually for retrieval.
# Step 4: At retrieval time, return the windowed context instead of just the sentence.

import re
from langchain_core.documents import Document

def build_sentence_window_chunks(text, window_size=2):
    """
    Splits text into sentences and wraps each with a context window.
    
    Args:
        text: The full document text.
        window_size: Number of sentences before and after to include as context.
    
    Returns:
        List of LangChain Documents where:
        - page_content = single sentence (for embedding)
        - metadata['window'] = sentence + surrounding context (for generation)
    """
    # Split on sentence-ending punctuation, keeping punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    docs = []
    for i, sentence in enumerate(sentences):
        # Define window boundaries
        start = max(0, i - window_size)
        end = min(len(sentences), i + window_size + 1)
        
        # Build the context window string
        window_text = " ".join(sentences[start:end])
        
        doc = Document(
            page_content=sentence,          # Only the sentence is embedded
            metadata={
                "sentence_id": i,
                "window": window_text,      # Full window stored as metadata
                "source": "ai_overview",
                "chunk_type": "sentence_window"
            }
        )
        docs.append(doc)
    
    return docs


sentence_window_chunks = build_sentence_window_chunks(sample_text, window_size=2)

print(f"Total sentence-level chunks: {len(sentence_window_chunks)}")
print(f"\nExample - Sentence (what gets embedded):")
print(f"  {sentence_window_chunks[20].page_content}")
print(f"\nExample - Window (what gets returned as context):")
print(f"  {sentence_window_chunks[20].metadata['window']}")

Total sentence-level chunks: 32

Example - Sentence (what gets embedded):
  LLMs suffer from hallucination, outdated knowledge, and context window limits, 
which is why RAG is used to augment them with external, up-to-date knowledge.

Example - Window (what gets returned as context):
  They can generate coherent, contextually relevant text. Examples include GPT-4, Claude, Llama, 
Gemini, and Mistral. LLMs suffer from hallucination, outdated knowledge, and context window limits, 
which is why RAG is used to augment them with external, up-to-date knowledge. Prompt engineering is the practice of designing inputs to AI models to achieve desired outputs. Effective prompts are clear, specific, and structured.


In [10]:
# Build a FAISS vector store using sentence-level embeddings.
# When retrieving, we substitute the full window text as context instead of just the sentence.

from langchain_community.vectorstores import FAISS

sw_vectorstore = FAISS.from_documents(sentence_window_chunks, embedding_model)

def sentence_window_retrieve(query, k=3):
    """
    Retrieves top-k sentences by semantic similarity,
    but returns their surrounding window as context.
    """
    results = sw_vectorstore.similarity_search(query, k=k)
    
    # Replace the sentence with its window context for generation
    context_parts = []
    for doc in results:
        window = doc.metadata.get("window", doc.page_content)
        context_parts.append(window)
    
    return context_parts


# Test the sentence-window retriever
query = "How do embeddings enable semantic search?"
windows = sentence_window_retrieve(query, k=2)

print(f"Query: '{query}'")
print("\nRetrieved windows (with surrounding context):")
for i, w in enumerate(windows):
    print(f"\nWindow {i+1}:")
    print(w)

Query: 'How do embeddings enable semantic search?'

Retrieved windows (with surrounding context):

Window 1:
Embeddings are numerical representations of data in a high-dimensional vector space. Similar items have embeddings that are close together in this space, enabling semantic similarity 
comparison. Text embeddings capture the meaning of words and sentences. Models like BERT, 
Sentence-BERT, and OpenAI Ada produce state-of-the-art text embeddings. Autonomous agents are AI systems that can perceive their environment, make decisions, and take 
actions to achieve goals without continuous human intervention.

Window 2:
Good prompt engineering 
significantly improves the quality and reliability of LLM outputs. Embeddings are numerical representations of data in a high-dimensional vector space. Similar items have embeddings that are close together in this space, enabling semantic similarity 
comparison. Text embeddings capture the meaning of words and sentences. Models like BERT, 
Senten

---

## 3. Advanced RAG Patterns

Once you have a basic RAG pipeline, there are several architectural patterns to extend and improve it. Each pattern addresses a specific limitation of simple RAG.

| Pattern | Problem It Solves |
|---|---|
| RAG with Memory | Cannot maintain multi-turn conversation context |
| HyDE | Query-document semantic mismatch |
| Multi-Query RAG | Single query misses relevant documents |
| Branched RAG | Different questions need different retrieval paths |
| Multimodal RAG | Documents contain images, charts, or tables |

### 3.1 Simple RAG (Baseline - Quick Recap)

This is the foundation. We set this up now so all other patterns can be compared against it.

In [11]:
# Build a baseline FAISS vector store using recursive chunks.
# This is our simple RAG foundation - all advanced patterns build on top of this.

from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, SystemMessage

# Create FAISS vector store from recursive chunks
base_vectorstore = FAISS.from_documents(recursive_chunks, embedding_model)      # recursive_chunks - Can refer to section 2.1 for how recursive_chunks were created.

def simple_rag(query, k=3):
    """
    Basic RAG: retrieve relevant chunks, then generate an answer.
    
    Steps:
    1. Embed the query
    2. Find top-k similar chunks in FAISS
    3. Build a prompt with retrieved context
    4. Call the LLM to generate an answer
    """
    # Step 1 & 2: Retrieve
    docs = base_vectorstore.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Step 3 & 4: Generate
    messages = [
        SystemMessage(content=(
            "You are a helpful assistant. Answer questions using only the provided context. "
            "If the answer is not in the context, say 'I don't know based on the provided information.'"
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    
    response = llm.invoke(messages)
    return response.content


# Test simple RAG
answer = simple_rag("What is the difference between machine learning and deep learning?")
print("Simple RAG Answer:")
print(answer)

Simple RAG Answer:
Based on the provided context, machine learning (ML) is a subset of artificial intelligence that involves building a model based on sample data to make predictions or decisions without explicit programming. 

Deep learning, on the other hand, is a part of a broader family of machine learning methods based on artificial neural networks with representation learning. This suggests that deep learning is a specific type of machine learning that uses neural networks, whereas machine learning is a broader field that encompasses various methods, including deep learning.


### 3.2 RAG with Conversation Memory

Simple RAG treats every question independently. In a real chatbot, the user might ask follow-up questions like "Tell me more about that" so the system needs to remember previous exchanges.

We maintain a conversation history and include it in the prompt so the LLM has context from previous turns.

In [12]:
# RAG with Conversation Memory.
# We maintain a chat history list and pass it to the LLM with every new query.
# The LLM sees both the retrieved context AND previous conversation turns.

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

class ConversationalRAG:
    """
    A RAG system that maintains conversation history across multiple turns.
    
    This allows users to ask follow-up questions that refer to previous exchanges.
    """
    
    def __init__(self, vectorstore, llm, k=3, max_history_turns=5):
        self.vectorstore = vectorstore
        self.llm = llm
        self.k = k
        self.max_history_turns = max_history_turns  # Limit history to avoid context overflow
        self.chat_history = []  # Stores (human_msg, ai_msg) tuples
    
    def chat(self, user_query):
        """
        Process a user query with conversation history and retrieved context.
        """
        # Step 1: Retrieve relevant context
        docs = self.vectorstore.similarity_search(user_query, k=self.k)
        context = "\n\n".join([doc.page_content for doc in docs])
        
        # Step 2: Build the message list with history
        messages = [
            SystemMessage(content=(
                "You are a knowledgeable assistant. Use the provided context to answer questions. "
                "You also have access to the conversation history. "
                "If a question refers to something mentioned earlier, use the history. "
                "If the answer is not in the context, say so honestly."
            ))
        ]
        
        # Add recent conversation history (last N turns)
        recent_history = self.chat_history[-self.max_history_turns:]
        for human_turn, ai_turn in recent_history:
            messages.append(HumanMessage(content=human_turn))
            messages.append(AIMessage(content=ai_turn))
        
        # Add the current query with retrieved context
        messages.append(HumanMessage(
            content=f"Context from knowledge base:\n{context}\n\nCurrent question: {user_query}"
        ))
        
        # Step 3: Generate response
        response = self.llm.invoke(messages)
        answer = response.content
        
        # Step 4: Save this turn to history
        self.chat_history.append((user_query, answer))
        
        return answer
    
    def reset(self):
        """Clear conversation history to start a new session."""
        self.chat_history = []
        print("Conversation history cleared.")


# Initialize the conversational RAG system
conv_rag = ConversationalRAG(vectorstore=base_vectorstore, llm=llm)

# Simulate a multi-turn conversation
print("=" * 60)
print("Turn 1")
print("=" * 60)
q1 = "What is deep learning?"
a1 = conv_rag.chat(q1)
print(f"User: {q1}")
print(f"Assistant: {a1}")

print("\n" + "=" * 60)
print("Turn 2 — Follow-up that refers to previous answer")
print("=" * 60)
q2 = "What are some applications of what you just described?"
a2 = conv_rag.chat(q2)
print(f"User: {q2}")
print(f"Assistant: {a2}")

Turn 1
User: What is deep learning?
Assistant: Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning.

Turn 2 — Follow-up that refers to previous answer
User: What are some applications of what you just described?
Assistant: Based on the context provided, some potential applications of deep learning, prompt engineering, and other related concepts include:

1. **Virtual Assistants**: Autonomous agents can be used to build virtual assistants like Siri, Alexa, or Google Assistant, which can perceive their environment, make decisions, and take actions to achieve goals without continuous human intervention.

2. **Chatbots and Customer Service**: Effective prompt engineering can be used to design inputs to AI models to achieve desired outputs, such as generating human-like responses to customer inquiries. This can improve the quality and reliability of chatbots and customer service systems.

3. **Natural Languag

### 3.3 HyDE - Hypothetical Document Embeddings

**Problem with simple RAG:** A user's question and the relevant document passage may be worded very differently. For example, the question is "Why does RAG reduce hallucination?" but the document says "RAG grounds responses in retrieved context." The embeddings of these may not be close enough.

**HyDE solution:** Instead of embedding the raw question, first ask the LLM to generate a *hypothetical answer* to the question. Then embed that hypothetical answer. This hypothetical answer is much closer in embedding space to the actual documents, improving retrieval.

```
User Query  ->  LLM generates hypothetical answer  ->  Embed hypothetical answer  ->  Retrieve similar docs
```

In [13]:
# HyDE: Hypothetical Document Embeddings
# Step 1: Use LLM to generate a hypothetical answer to the query
# Step 2: Embed the hypothetical answer (not the original query)
# Step 3: Use this richer embedding to search the vector store
# Step 4: Retrieve real documents, use them for final generation

from langchain_core.messages import HumanMessage, SystemMessage

def generate_hypothetical_document(query):
    """
    Ask the LLM to write a short passage that would answer the query,
    even without access to the actual knowledge base.
    This hypothetical passage is then used for embedding-based retrieval.
    """
    messages = [
        SystemMessage(content=(
            "Write a short, factual paragraph (2-3 sentences) that answers the following question. "
            "Write it as if it came from a technical document. Do not add any preamble — just the paragraph."
        )),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)
    return response.content


def hyde_rag(query, vectorstore, k=3):
    """
    Full HyDE pipeline:
    1. Generate a hypothetical answer to the query
    2. Embed the hypothetical answer
    3. Search the vector store with that embedding
    4. Generate the real answer using retrieved documents
    """
    # Step 1: Generate hypothetical document
    hypothetical_doc = generate_hypothetical_document(query)
    print(f"Hypothetical document generated:\n  {hypothetical_doc[:400]}...\n")
    
    # Step 2 & 3: Use the hypothetical document for retrieval
    # The embedding of the hypothetical answer is closer to real documents than the bare question
    docs = vectorstore.similarity_search(hypothetical_doc, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Step 4: Generate the real answer using retrieved context
    messages = [
        SystemMessage(content=(
            "Answer the question using only the provided context. "
            "Be precise and factual."
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    response = llm.invoke(messages)
    return response.content, docs


# Test HyDE
query = "Why is RAG better than fine-tuning for keeping LLMs up to date?"

print(f"Query: {query}\n")
answer, retrieved_docs = hyde_rag(query, base_vectorstore)
print("HyDE RAG Answer:")
print(answer)

Query: Why is RAG better than fine-tuning for keeping LLMs up to date?

Hypothetical document generated:
  RAG (Retrieve-Augment-Generate) is considered a more effective method than fine-tuning for keeping large language models (LLMs) up to date due to its ability to leverage external knowledge sources, such as databases and APIs, to augment the model's knowledge without requiring extensive retraining. This approach allows for more efficient and targeted updates, reducing the risk of catastrophic forge...

HyDE RAG Answer:
RAG is better than fine-tuning for keeping LLMs up to date because it grounds the LLM response in retrieved context from an external knowledge base, rather than relying solely on parametric memory. This reduces hallucination by using the most accurate, up-to-date information available.


### 3.4 Multi-Query RAG

**Problem:** A single query may miss relevant documents because documents are phrased differently. One query only covers one perspective.

**Solution:** Generate multiple different phrasings of the same question, retrieve documents for each, and merge the results. This increases recall hence more relevant documents are found.

```
Original Query
    ├── Rephrasing 1 --> Retrieved Docs A
    ├── Rephrasing 2 --> Retrieved Docs B   ---> Deduplicated docs --> LLM
    └── Rephrasing 3 --> Retrieved Docs C
```

In [14]:
# Multi-Query RAG
# Step 1: Use LLM to generate N alternative phrasings of the query
# Step 2: Run retrieval for each phrasing
# Step 3: Deduplicate documents across all retrievals (This means if the same document is retrieved for multiple query variations, we only keep it once in the final context)
# Step 4: Generate answer from the merged, deduplicated context (This means we combine all unique retrieved documents into one context and pass it to the LLM to generate a single, comprehensive answer.)

def generate_query_variations(query, n=3):
    """
    Ask the LLM to generate N alternative phrasings of the original query.
    Different phrasings will match different documents in the vector store.
    """
    messages = [
        SystemMessage(content=(
            f"Generate {n} different phrasings of the following question. "
            "Each rephrasing should approach the topic from a slightly different angle. "
            f"Output exactly {n} lines, one rephrasing per line, no numbering, no extra text."
        )),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)
    # Parse the response into a list of query strings
    variations = [line.strip() for line in response.content.strip().split("\n") if line.strip()]
    return variations[:n]  # Safety: take exactly n


def multi_query_rag(query, vectorstore, n_queries=3, k=3):
    """
    Retrieve documents using multiple query phrasings and merge results.
    Deduplication is done based on document content.
    """
    # Step 1: Generate query variations
    all_queries = [query] + generate_query_variations(query, n=n_queries)
    
    print(f"Generated {len(all_queries)} query variations:")
    for i, q in enumerate(all_queries):
        print(f"  Query {i+1}: {q}")
    print()
    
    # Step 2: Retrieve for each query
    seen_contents = set()  # Track unique document contents
    unique_docs = []
    
    for q in all_queries:
        docs = vectorstore.similarity_search(q, k=k)
        for doc in docs:
            content_hash = hash(doc.page_content)  # Use content hash for deduplication
            if content_hash not in seen_contents:
                seen_contents.add(content_hash)
                unique_docs.append(doc)
    
    print(f"Unique documents retrieved: {len(unique_docs)}")
    
    # Step 3: Build context from all unique docs
    context = "\n\n".join([doc.page_content for doc in unique_docs])
    
    # Step 4: Generate final answer
    messages = [
        SystemMessage(content=(
            "Answer the question based only on the provided context. Be comprehensive."
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    response = llm.invoke(messages)
    return response.content


# Test Multi-Query RAG
query = "How do vector databases support AI applications?"
print(f"Original Query: {query}\n")
answer = multi_query_rag(query, base_vectorstore)
print("Multi-Query RAG Answer:")
print(answer)

Original Query: How do vector databases support AI applications?

Generated 4 query variations:
  Query 1: How do vector databases support AI applications?
  Query 2: What role do vector databases play in enabling the development and deployment of artificial intelligence systems?
  Query 3: How can vector databases be leveraged to improve the performance and efficiency of AI-powered applications?
  Query 4: In what ways do vector databases facilitate the integration of AI algorithms with large-scale, high-dimensional data sets?

Unique documents retrieved: 4
Multi-Query RAG Answer:
Vector databases support AI applications in several ways:

1. **Enabling semantic search**: Vector databases allow for fast similarity search over large document collections, which is fundamental to Recommender AI systems (RAG systems). This enables AI applications to retrieve relevant information from vast amounts of data.

2. **Facilitating efficient similarity search**: By storing data as high-dimensional

### 3.5 Branched RAG (Query Routing)

In production systems, not all queries are the same type. Some might be factual lookups, others might be analytical questions, and some might need real-time data. Branched RAG routes queries to the most appropriate retrieval pathway.

**Architecture:**
```
User Query
    |
    v
Router (LLM decides path)
    ├── factual    --> Dense Vector Search
    ├── analytical --> Multi-Query RAG
    └── general    --> Simple RAG
```

In [15]:
# Branched RAG - Query Routing
# The router LLM classifies the query, then dispatches to the right retrieval path.
# This allows different strategies to be optimized for different question types.

import json

def route_query(query):
    """
    Classify the query type using the LLM.
    Returns one of: 'factual', 'analytical', 'general'
    """
    messages = [
        SystemMessage(content=(
            "Classify the following question into exactly one of these categories: "
            "factual, analytical, general. "
            "- factual: asks for a specific fact, definition, or named entity (e.g. 'What is FAISS?') "
            "- analytical: asks for comparison, explanation of cause/effect, or reasoning (e.g. 'Why is RAG better than X?') "
            "- general: a broad or open-ended question "
            "Reply with ONLY the category word, nothing else."
        )),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)
    category = response.content.strip().lower()
    
    # Safety: default to general if unexpected output
    if category not in ["factual", "analytical", "general"]:
        category = "general"
    return category


def branched_rag(query, vectorstore):
    """
    Routes the query to the appropriate retrieval strategy based on query type.
    """
    # Step 1: Classify the query
    query_type = route_query(query)
    print(f"Query type detected: '{query_type}'")
    
    # Step 2: Route to the appropriate pipeline
    if query_type == "factual":
        # Factual queries: precise single-query retrieval with more chunks
        print("Routing to: Dense vector search (factual lookup)")
        docs = vectorstore.similarity_search(query, k=2)
        context = "\n\n".join([doc.page_content for doc in docs])
        prompt = f"Answer this factual question concisely using only the context.\n\nContext:\n{context}\n\nQuestion: {query}"
        messages = [HumanMessage(content=prompt)]
    
    elif query_type == "analytical":
        # Analytical queries: use HyDE for richer retrieval
        print("Routing to: HyDE-enhanced retrieval (analytical question)")
        answer, _ = hyde_rag(query, vectorstore)
        return answer  # HyDE already generates the final answer
    
    else:
        # General queries: standard RAG
        print("Routing to: Standard RAG (general question)")
        docs = vectorstore.similarity_search(query, k=4)
        context = "\n\n".join([doc.page_content for doc in docs])
        prompt = f"Answer the following question using the provided context.\n\nContext:\n{context}\n\nQuestion: {query}"
        messages = [HumanMessage(content=prompt)]
    
    response = llm.invoke(messages)
    return response.content


# Test Branched RAG with different query types
queries = [
    "What is FAISS?",                                         # Should route to: factual
    "Why does RAG perform better than pure LLM generation?",  # Should route to: analytical
    "Tell me about AI and its subfields.",                    # Should route to: general
]

for q in queries:
    print("\n" + "=" * 60)
    print(f"Query: {q}")
    print("=" * 60)
    answer = branched_rag(q, base_vectorstore)
    print(f"Answer: {answer[:300]}..." if len(answer) > 300 else f"Answer: {answer}")


Query: What is FAISS?
Query type detected: 'factual'
Routing to: Dense vector search (factual lookup)
Answer: FAISS is an open-source library for efficient similarity search.

Query: Why does RAG perform better than pure LLM generation?
Query type detected: 'analytical'
Routing to: HyDE-enhanced retrieval (analytical question)
Hypothetical document generated:
  RAG (Retrieve-Augment-Generate) models perform better than pure LLM (Large Language Model) generation due to their ability to leverage external knowledge sources, such as large-scale databases or web corpora, to retrieve relevant information and augment the generated text. This approach allows RAG models to tap into a vast amount of domain-specific knowledge, reducing the likelihood of generating ...

Answer: RAG performs better than pure LLM generation because it reduces hallucination by grounding the LLM response in retrieved context rather than relying solely on parametric memory. This is achieved by combining dense retrieva

### 3.6 Multimodal RAG

Standard RAG only processes text. But real documents contain images, charts, diagrams, and tables that are crucial for understanding. Multimodal RAG allows the system to reason over both text and images.

**Architecture:**
```
Document (text + images)
    |
    v
Image caption generation (Vision LLM) + Text chunking
    |
    v
All content (text + captions) stored in vector store
    |
    v
Query -> Retrieve text/image content -> Multimodal LLM generates answer
```

We use **Gemini-3-Flash** (free tier) as it is a multimodal model that can process both text and images. 

In [26]:
# Multimodal RAG - Part 1: Setup Gemini
# Gemini 1.5 Flash: free tier, supports text + image input

import google.generativeai as genai

# Configure Gemini with your API key from .env
genai.configure(api_key=GOOGLE_API_KEY)

# Use gemini-2.5-flash  fast, free, and multimodal
# gemini_model = genai.GenerativeModel("gemini-2.5-flash")
gemini_model = genai.GenerativeModel("gemini-3-flash-preview")

print("Gemini 3 Flash Preview model initialized.")

Gemini 3 Flash Preview model initialized.


In [21]:
# Multimodal RAG - Part 2: Process an image
# In a real pipeline, you would extract images from PDFs, screenshots, etc.
# Here we load a local image to demonstrate the concept.

from PIL import Image
import os

def describe_image_with_gemini(image_path, description_prompt="Describe this image in detail for use in a knowledge base."):
    """
    Uses Gemini Vision to generate a text description of an image.
    This description can then be embedded and stored in the vector store,
    making image content searchable via text queries.
    """
    image = Image.open(image_path)
    image.thumbnail((1024, 1024))       # Optional: Resize if the image is huge

    response = gemini_model.generate_content([image, description_prompt])
    return response.text


def answer_question_about_image(image_path, question):
    """
    Directly answers a question about an image using Gemini Vision.
    This is used when the retrieved content is an image chunk.
    """
    image = Image.open(image_path)
    response = gemini_model.generate_content([image, question])
    return response.text


# Usage Example:
image_path = "demo_image.jpg"
caption = describe_image_with_gemini(image_path)
print("Generated caption:", caption)

# Once you have captions, add them as Documents to your vector store:
image_doc = Document(
    page_content=caption,
    metadata={"source": image_path, "type": "image"}
)
# Then when this doc is retrieved, pass the image to Gemini for final QA.

print("Multimodal helper functions defined.")
print("To use: provide an image path and call describe_image_with_gemini() or answer_question_about_image()")

Generated caption: This image is a detailed, close-up, and whimsical depiction of an orange tabby cat wearing a futuristic astronaut suit in what appears to be outer space.

**Main Subject:**
*   **The Cat:** The central focus is a domestic short-haired cat with prominent orange/ginger fur and distinct darker tabby stripes visible on its forehead, cheeks, and around its muzzle. Its most striking features are its large, round, bright blue eyes, which are wide-set and gaze directly forward with an expression of curiosity, wonder, or slight bewilderment. A small, pink, triangular nose sits below its eyes, and numerous long, white whiskers fan out from its cheeks. Its ears are pointed and upright, with lighter fur visible inside.
*   **The Astronaut Suit:** The cat's head is perfectly enclosed within a clear, reflective, domed visor of a pristine white astronaut helmet. The suit itself is robust and appears to be made of a sturdy, possibly slightly textured material, predominantly white. V

In [27]:
# Multimodal RAG - Part 3: Full Pipeline Architecture
# This class shows how multimodal RAG works end-to-end.
# It handles documents that contain both text chunks and image chunks.

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

class MultimodalRAG:
    """
    A RAG system that handles both text and image documents.
    
    Ingestion:
    - Text: chunked and embedded as usual
    - Images: captioned by Gemini, captions are embedded
    
    Retrieval:
    - Query matches both text chunks and image captions
    - If an image chunk is retrieved, Gemini processes it directly
    """
    
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.documents = []  # All documents (text + image captions)
        self.vectorstore = None
        self.image_registry = {}  # Maps doc_id -> image_path for image documents
    
    def add_text_documents(self, docs):
        """Add plain text documents to the knowledge base."""
        for doc in docs:
            doc.metadata["content_type"] = "text"
        self.documents.extend(docs)
    
    def add_image(self, image_path, doc_id=None):
        """
        Add an image to the knowledge base.
        Gemini generates a caption, which is then embedded for retrieval.
        """
        doc_id = doc_id or f"img_{len(self.image_registry)}"
        
        # Generate text caption from image using Gemini
        caption = describe_image_with_gemini(
            image_path,
            "Describe all visual elements, text, charts, and data in this image in detail."
        )
        
        # Store caption as a document
        doc = Document(
            page_content=caption,
            metadata={"doc_id": doc_id, "source": image_path, "content_type": "image"}
        )
        self.documents.append(doc)
        self.image_registry[doc_id] = image_path
        
        print(f"Image '{image_path}' ingested as doc_id '{doc_id}'")
        return caption
    
    def build_index(self):
        """Build the FAISS vector store from all ingested documents."""
        self.vectorstore = FAISS.from_documents(self.documents, self.embedding_model)
        print(f"Index built with {len(self.documents)} documents.")
    
    def query(self, user_query, k=3):
        """
        Answer a query using both text and image knowledge.
        """
        retrieved = self.vectorstore.similarity_search(user_query, k=k)
        
        text_context_parts = []
        image_answers = []
        
        for doc in retrieved:
            if doc.metadata.get("content_type") == "image":
                # For image chunks: send the image + question directly to Gemini
                doc_id = doc.metadata.get("doc_id")
                if doc_id in self.image_registry:
                    img_path = self.image_registry[doc_id]
                    img_answer = answer_question_about_image(img_path, user_query)
                    image_answers.append(img_answer)
            else:
                # For text chunks: add to context as usual
                text_context_parts.append(doc.page_content)
        
        # Combine text context and any image-based answers
        full_context = "\n\n".join(text_context_parts)
        if image_answers:
            full_context += "\n\nFrom images in knowledge base:\n" + "\n".join(image_answers)
        
        # Final generation with combined context
        messages = [
            SystemMessage(content="Answer based on the provided context which may include text and image descriptions."),
            HumanMessage(content=f"Context:\n{full_context}\n\nQuestion: {user_query}")
        ]
        response = llm.invoke(messages)
        return response.content


# print("MultimodalRAG class defined.")
# print("\nUsage:")
# print("  mm_rag = MultimodalRAG(embedding_model)")
# print("  mm_rag.add_text_documents(recursive_chunks)")
# print("  mm_rag.add_image('path/to/chart.png')  # Requires GOOGLE_API_KEY")
# print("  mm_rag.build_index()")
# print("  answer = mm_rag.query('What does the chart show?')")

print("MultimodalRAG class defined.")
print("\nUsage:")
mm_rag = MultimodalRAG(embedding_model)
mm_rag.add_text_documents(recursive_chunks)
mm_rag.add_image('demo_image.jpg')
mm_rag.build_index()
answer = mm_rag.query('How has RAG or AI helped in creating any image like Cat in Space Generated image ?')
answer

MultimodalRAG class defined.

Usage:
Image 'demo_image.jpg' ingested as doc_id 'img_0'
Index built with 11 documents.


'According to the provided context, RAG or AI has helped in creating the Cat in Space generated image in the following ways:\n\n1.  **Detailing:** RAG helped the model understand the complex textures of the fabric and the mechanical details of the helmet.\n2.  **Lighting/Composition:** The AI used its training to create the "cinematic" feel, using "retrieved" knowledge of how professional photography uses depth of field (the blurred background) to make the kitten the focal point.\n3.  **Anatomy:** AI ensures the kitten\'s eyes are expressive and anatomically correct, while the "space" context is accurately reflected in the visor.\n\nAdditionally, RAG-like processes likely helped in the following ways:\n\n*   **Enhanced Prompting (LLM-assisted RAG):** A RAG-enabled LLM might have looked up specific details about modern spacesuits or cinematic lighting styles to create a much more detailed prompt for the image engine.\n*   **Retrieval-Augmented Image Generation (RA-IG):** The AI might ha

---

## 4. Hybrid Search and Retrieval

There are two fundamentally different ways to search documents:

| Method | How It Works | Strength | Weakness |
|---|---|---|---|
| Dense (Vector) | Embeds text as vectors, finds semantically similar | Understands meaning | Misses exact keyword matches |
| Sparse (BM25) | Keyword frequency ranking (like search engines) | Exact keyword matches | No semantic understanding |

**Hybrid Search** combines both. When a user asks about "FAISS", BM25 finds exact matches. When they ask "how do vector databases help in AI?", dense retrieval catches the semantic match. Together they cover more ground.

We combine results using **Reciprocal Rank Fusion (RRF)** - a simple but powerful method to merge ranked lists from different sources.


```
1. To understand why Hybrid is king, we have to see where the others fail:
Vector Search is "Too Smart": If we search for a specific product ID like "SKU-9921", a vector search might return other products because they "look similar" in meaning, even if the ID is totally different.
Keyword Search is "Too Literal": If we search for "feline medication", BM25 will completely miss a document that says "cat medicine" because the words don't match exactly.
Hybrid Search acts like a search engine that is both a Librarian (understands meaning) and an Index Clerk (finds exact words).

2. How do we merge them? (The RRF Logic)
When we search using two methods, we get two different lists. Reciprocal Rank Fusion (RRF) is the "judge" that decides the final order.
The Rule: A document’s score is calculated by the inverse of its rank.
If a document is #1 in a list, it gets a high score (1/1).
If it’s #100, it gets a tiny score (1/100).
Why this works: If a document appears in the top 5 of both lists, its combined score will be much higher than a document that is #1 in one list but totally missing from the other. It rewards consistency.

3. The Hybrid Workflow
User Query: "Best medicine for a sick cat."
Dense Search: Finds "feline health," "pet clinics," "cat care."
Sparse Search: Finds "medicine," "best," "sick," "cat."
RRF Merge: Realizes that "cat health and medicine" appeared in both lists and pushes it to the #1 spot.

4. When to use Hybrid Search?
Domain-Specific Jargon: When the data has weird acronyms or part numbers (e.g., Medical, Legal, or Tech support).
Short Queries: When users type only 1 or 2 words, where semantic meaning is hard to guess.

```

In [30]:
# Build BM25 sparse retriever
# BM25 is a classic information retrieval algorithm based on term frequency and inverse document frequency.
# It excels at exact keyword matching - essential when queries contain specific technical terms.

from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize all chunks for BM25
# BM25 works at the word level, not the embedding level
corpus_texts = [chunk.page_content for chunk in recursive_chunks]
tokenized_corpus = [text.lower().split() for text in corpus_texts]

bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, k=5):
    """
    Search using BM25 keyword matching.
    Returns top-k documents with their BM25 scores.
    """
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    # Get indices of top-k highest scores
    top_indices = np.argsort(scores)[::-1][:k]
    
    results = []
    for idx in top_indices:
        doc = recursive_chunks[idx]
        results.append((doc, float(scores[idx]), int(idx)))
    
    return results  # Returns list of (document, score, index)


# Test BM25
query = "FAISS similarity search"
bm25_results = bm25_search(query, k=3)

print(f"BM25 results for: '{query}'")
for i, (doc, score, idx) in enumerate(bm25_results):
    print(f"\nResult {i+1} (score={score:.4f}, chunk_id={idx}):")
    # print(f"  {doc.page_content[:150]}...")
    print(f"  {doc.page_content}")


BM25 results for: 'FAISS similarity search'

Result 1 (score=5.8513, chunk_id=5):
  Vector databases store data as high-dimensional vectors, enabling fast similarity search. 
Popular vector databases include FAISS, Pinecone, Weaviate, Chroma, and Qdrant. They are 
fundamental to RAG systems because they allow semantic search over large document collections. 
FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search.

Result 2 (score=1.2887, chunk_id=8):
  Embeddings are numerical representations of data in a high-dimensional vector space. 
Similar items have embeddings that are close together in this space, enabling semantic similarity 
comparison. Text embeddings capture the meaning of words and sentences. Models like BERT, 
Sentence-BERT, and OpenAI Ada produce state-of-the-art text embeddings.

Result 3 (score=0.0000, chunk_id=9):
  Autonomous agents are AI systems that can perceive their environment, make decisions, and take 
actions to achieve

In [31]:
# Reciprocal Rank Fusion (RRF) implementation
# RRF is a simple but effective way to merge ranked lists from different retrieval methods.
# Formula: score(doc) = sum over retrievers of 1 / (k + rank)
# where k=60 is a constant that smooths the scores and reduces the impact of high ranks.
# Documents appearing high in multiple lists get the highest combined scores.

def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Merge multiple ranked lists of documents using Reciprocal Rank Fusion.
    
    Args:
        ranked_lists: List of lists of (document, score) tuples, each sorted by relevance
        k: Smoothing constant (default 60, as used in the original RRF paper)
    
    Returns:
        List of documents sorted by their combined RRF score
    """
    # Accumulate RRF scores per document (keyed by content hash)
    rrf_scores = {}
    doc_map = {}  # Store document objects
    
    for ranked_list in ranked_lists:
        for rank, (doc, score) in enumerate(ranked_list):
            doc_key = hash(doc.page_content)  # Use content as unique identifier
            
            # RRF formula: higher rank (lower rank number) = higher contribution
            rrf_contribution = 1.0 / (k + rank + 1)
            
            rrf_scores[doc_key] = rrf_scores.get(doc_key, 0) + rrf_contribution
            doc_map[doc_key] = doc
    
    # Sort by RRF score (descending)
    sorted_doc_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    
    return [
        (doc_map[key], rrf_scores[key]) 
        for key in sorted_doc_keys
    ]


print("RRF function defined.")

RRF function defined.


In [33]:
# Full Hybrid Search Pipeline
# Combines BM25 (keyword) + FAISS (semantic) with RRF fusion

def hybrid_search(query, vectorstore, k=5, final_k=4):
    """
    Hybrid retrieval: combines dense (FAISS) and sparse (BM25) retrieval.
    
    Args:
        query: User's query string
        vectorstore: FAISS vector store (dense retrieval)
        k: Number of results to fetch from each retriever
        final_k: Number of final results to return after fusion
    
    Returns:
        List of top documents after hybrid scoring
    """
    # Dense retrieval: FAISS semantic search
    dense_docs = vectorstore.similarity_search(query, k=k)
    dense_ranked = [(doc, 1.0) for doc in dense_docs]  # Score is implicit by rank position
    
    # Sparse retrieval: BM25 keyword search
    bm25_results = bm25_search(query, k=k)
    bm25_ranked = [(doc, score) for doc, score, _ in bm25_results]
    
    # Fuse the two ranked lists using RRF
    fused_results = reciprocal_rank_fusion([dense_ranked, bm25_ranked])
    
    # Return only top final_k documents
    return [doc for doc, score in fused_results[:final_k]]


def hybrid_rag(query, vectorstore):
    """
    Full RAG pipeline using hybrid search for retrieval.
    """
    # Retrieve using hybrid search
    docs = hybrid_search(query, vectorstore)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Generate answer
    messages = [
        SystemMessage(content=(
            "You are a helpful assistant. Answer using only the provided context. "
            "If the answer is not present, say so."
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    response = llm.invoke(messages)
    return response.content


# Test Hybrid RAG vs Simple RAG
test_queries = [
    "What is FAISS?",                         # BM25 advantage: exact keyword
    "How does semantic similarity work?",      # Dense advantage: conceptual
]

for query in test_queries:
    print("\n" + "=" * 60)
    print(f"Query: {query}")
    print("=" * 60)
    
    simple_answer = simple_rag(query)
    hybrid_answer = hybrid_rag(query, base_vectorstore)
    
    # print(f"Simple RAG: {simple_answer[:200]}...")
    # print(f"\nHybrid RAG: {hybrid_answer[:200]}...")

    print(f"Simple RAG: {simple_answer}")
    print(f"\nHybrid RAG: {hybrid_answer}")    


Query: What is FAISS?
Simple RAG: FAISS is an open-source library for efficient similarity search.

Hybrid RAG: FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search.

Query: How does semantic similarity work?
Simple RAG: Semantic similarity comparison is enabled by embeddings that are close together in a high-dimensional vector space. Similar items have embeddings that are close together, allowing for the comparison of semantic similarity between them.

Hybrid RAG: Semantic similarity comparison is enabled by embeddings, which are numerical representations of data in a high-dimensional vector space. Similar items have embeddings that are close together in this space, allowing for the comparison of semantic similarity.


---

## 5. Reranking Models

**The retrieval problem:** Vector search finds documents that are *semantically close* to the query, but not necessarily the most *relevant* to answer it. The top-5 retrieved documents might not be ranked in the right order.

**Reranking solution:** After initial retrieval (which is fast but approximate), apply a more expensive but more accurate model to score and rerank the candidates.

**Two-stage pipeline:**
```
Query --> Fast retrieval (top 20 candidates) --> Reranker (scores each candidate) --> Top 3 --> LLM
```

**Cross-encoder vs Bi-encoder:**
- **Bi-encoder** (FAISS): Embeds query and document independently. Fast, scalable, but less accurate.
- **Cross-encoder** (Reranker): Processes query and document together. Slow, not scalable alone, but much more accurate.

We use `cross-encoder/ms-marco-MiniLM-L-6-v2` — a free, local reranking model.

In [ ]:
# Load the cross-encoder reranking model.
# CrossEncoder processes (query, document) pairs together - it understands how the query and document relate to each other, not just independently.
# This makes it much more accurate than bi-encoder similarity alone.

from sentence_transformers import CrossEncoder

# ms-marco-MiniLM-L-6-v2: trained on MS MARCO passage ranking dataset
# It outputs a relevance score for each (query, passage) pair
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-encoder reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")
print("This model runs locally - no API key needed.")

d:\Work\Learning\Topics_by_Abhay_Sir\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\siddhant.khobragade\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3458.72it/s]
BertFo

Cross-encoder reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
This model runs locally — no API key needed.


In [35]:
# Demonstrate what the reranker does.
# It scores each (query, document) pair on a scale from low to high relevance.

query = "How does RAG reduce hallucination?"

sample_passages = [
    "RAG reduces hallucination by grounding responses in retrieved context rather than relying solely on parametric memory.",
    "Deep learning architectures include convolutional neural networks and recurrent neural networks.",
    "RAG systems combine dense retrieval with generative models to produce accurate answers.",
    "Vector databases store data as high-dimensional vectors, enabling fast similarity search.",
]

# Score each passage against the query
pairs = [(query, passage) for passage in sample_passages]
scores = reranker.predict(pairs)

print(f"Reranker scores for query: '{query}'\n")
ranked = sorted(zip(scores, sample_passages), reverse=True)
for score, passage in ranked:
    # print(f"Score {score:.4f}: {passage[:100]}...")
    print(f"Score {score:.4f}: {passage}")

# Observation: passages directly answering the question get higher scores

Reranker scores for query: 'How does RAG reduce hallucination?'

Score 9.3769: RAG reduces hallucination by grounding responses in retrieved context rather than relying solely on parametric memory.
Score -5.9916: RAG systems combine dense retrieval with generative models to produce accurate answers.
Score -11.3857: Deep learning architectures include convolutional neural networks and recurrent neural networks.
Score -11.4022: Vector databases store data as high-dimensional vectors, enabling fast similarity search.


In [36]:
# Full Retrieve-Then-Rerank-Then-Generate pipeline.
# This is the production-grade approach used in most serious RAG systems.

def retrieve_and_rerank(query, vectorstore, initial_k=10, final_k=3):
    """
    Two-stage retrieval with reranking.
    
    Stage 1 (fast): Retrieve initial_k candidates using FAISS vector search
    Stage 2 (accurate): Rerank candidates using cross-encoder, return top final_k
    
    Args:
        query: User query
        vectorstore: FAISS vector store
        initial_k: How many candidates to retrieve initially (cast a wide net)
        final_k: How many to return after reranking (precision)
    """
    # Stage 1: Broad retrieval - retrieve more candidates than we need
    initial_docs = vectorstore.similarity_search(query, k=initial_k)
    
    # Stage 2: Rerank - score each candidate against the query
    pairs = [(query, doc.page_content) for doc in initial_docs]
    rerank_scores = reranker.predict(pairs)
    
    # Sort by reranker score (descending) and take top final_k
    scored_docs = list(zip(rerank_scores, initial_docs))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    top_docs = [doc for score, doc in scored_docs[:final_k]]
    top_scores = [score for score, doc in scored_docs[:final_k]]
    
    return top_docs, top_scores


def rag_with_reranking(query, vectorstore):
    """
    Full RAG pipeline with retrieve -> rerank -> generate.
    """
    # Retrieve and rerank
    docs, scores = retrieve_and_rerank(query, vectorstore, initial_k=8, final_k=3)
    
    print(f"Top documents after reranking (scores: {[f'{s:.3f}' for s in scores]}):")
    for i, (doc, score) in enumerate(zip(docs, scores)):
        # print(f"  Rank {i+1} (score={score:.4f}): {doc.page_content[:100]}...")
        print(f"  Rank {i+1} (score={score:.4f}): {doc.page_content}")
    print()
    
    # Build context from reranked docs
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Generate answer
    messages = [
        SystemMessage(content=(
            "Answer the question using only the provided context. Be concise and precise."
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    response = llm.invoke(messages)
    return response.content


# Test and compare: No reranking vs With reranking
query = "What role do embeddings play in semantic search?"

print("=" * 60)
print("Without Reranking (Simple RAG):")
print("=" * 60)
print(simple_rag(query))

print("\n" + "=" * 60)
print("With Reranking:")
print("=" * 60)
print(rag_with_reranking(query, base_vectorstore))

Without Reranking (Simple RAG):
Embeddings play a crucial role in semantic search by enabling semantic similarity comparison. Similar items have embeddings that are close together in the high-dimensional vector space, allowing for the comparison of semantic similarity.

With Reranking:
Top documents after reranking (scores: ['5.135', '-4.870', '-8.892']):
  Rank 1 (score=5.1345): Embeddings are numerical representations of data in a high-dimensional vector space. 
Similar items have embeddings that are close together in this space, enabling semantic similarity 
comparison. Text embeddings capture the meaning of words and sentences. Models like BERT, 
Sentence-BERT, and OpenAI Ada produce state-of-the-art text embeddings.
  Rank 2 (score=-4.8703): Vector databases store data as high-dimensional vectors, enabling fast similarity search. 
Popular vector databases include FAISS, Pinecone, Weaviate, Chroma, and Qdrant. They are 
fundamental to RAG systems because they allow semantic search 

In [37]:
# Ultimate combined pipeline: Hybrid Search + Reranking + Generation
# This is production-grade RAG: the best retrieval + the most accurate reranking

def ultimate_rag(query, vectorstore, initial_k=12, final_k=3):
    """
    Production-grade RAG pipeline:
    1. Hybrid Search (Dense + BM25) -> initial candidates
    2. Cross-encoder Reranking -> best candidates
    3. LLM Generation -> final answer
    """
    # Stage 1: Hybrid retrieval (gets more diverse candidates than dense-only)
    hybrid_docs = hybrid_search(query, vectorstore, k=initial_k // 2, final_k=initial_k)
    
    # Stage 2: Rerank with cross-encoder
    pairs = [(query, doc.page_content) for doc in hybrid_docs]
    scores = reranker.predict(pairs)
    
    scored = sorted(zip(scores, hybrid_docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for score, doc in scored[:final_k]]
    
    # Stage 3: Generate
    context = "\n\n".join([doc.page_content for doc in top_docs])
    messages = [
        SystemMessage(content=(
            "You are an expert assistant. Answer accurately using only the provided context. "
            "If the answer is not in the context, say 'I cannot find this in the provided knowledge base.'"
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    response = llm.invoke(messages)
    return response.content


# Test the ultimate pipeline
query = "How do autonomous agents use LLMs for decision making?"
print(f"Query: {query}\n")
answer = ultimate_rag(query, base_vectorstore)
print("Ultimate RAG Answer:")
print(answer)

Query: How do autonomous agents use LLMs for decision making?

Ultimate RAG Answer:
Autonomous agents use LLMs as their reasoning core. This means that LLMs are used to make decisions and take actions to achieve goals without continuous human intervention.


---

## 6. Summary

This notebook covered the advanced layer of RAG, building on top of the basics.

**Chunking Strategies:**
- Recursive Character: General-purpose baseline
- Semantic: Splits on meaning changes, keeps topics together
- Hierarchical (Parent-Document): Embeds small chunks, returns large parent chunks
- Sentence-Window: Embeds individual sentences, returns surrounding context

**RAG Patterns:**
- Simple RAG: Baseline
- Conversational RAG: Maintains chat history for multi-turn dialogue
- HyDE: Generates hypothetical answer first to improve embedding match
- Multi-Query: Uses multiple phrasings to increase recall
- Branched (Router) RAG: Routes queries to different pipelines based on type
- Multimodal RAG: Handles images using Gemini for captioning and QA

**Hybrid Search:**
- Dense (FAISS): Semantic understanding
- Sparse (BM25): Exact keyword matching
- RRF Fusion: Combines both ranked lists into one final ranking

**Reranking:**
- Cross-encoder reranking: More accurate than bi-encoder, applied post-retrieval
- Two-stage pipeline: Broad retrieval -> accurate reranking -> generation

**Now next we will proceed to Notebook 3 for RAG Evaluation (RAGAS, DeepEval) and building a full RAG Chatbot.**